# Managed Agents session · Demo notebook

**Presenter copy.** Before the session: paste your API key and run every cell above **Demo starts here**. That creates the agent and environment and uploads the tickets, so the demo opens on the first run.

**Fallback:** after a full rehearsal, save a copy with its outputs as `demo_prerun.ipynb` (File → Save Notebook As). If the network fails live, open that copy and walk the same three beats.

## Setup (run before the session)

In [1]:
import os
from getpass import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("API key (starts with sk-ant-): ")

API key (starts with sk-ant-):  ········


In [2]:
import helpers

helpers.preflight()

✓ Python 3.14.7
✓ Anthropic SDK 1.8.0
✓ API key set
✓ tickets.json found
✓ Managed Agents API reachable with this key

Ready to go.


True

In [3]:
import json
from pathlib import Path

tickets = json.loads(Path("tickets.json").read_text())
for t in tickets:
    print(t["id"], t["customer_id"], "-", t["subject"])

T-1001 C-101 - Dashboard slow to load since Monday
T-1002 C-102 - SSO login failing for all users
T-1003 C-103 - UNACCEPTABLE!!! Dark mode colors
T-1004 C-104 - Small thing: CSV export dates
T-1005 C-101 - How do I add a teammate?
T-1006 C-105 - Charged twice this month
T-1007 C-999 - API returning 500 errors intermittently
T-1008 C-103 - Feature request: emoji reactions


In [4]:
CRM = {
    "C-101": {"name": "Northwind Logistics", "plan": "Growth",     "arr_usd": 18000,  "renewal_date": "2027-02-01"},
    "C-102": {"name": "Brightpath Health",   "plan": "Enterprise", "arr_usd": 240000, "renewal_date": "2027-03-15"},
    "C-103": {"name": "Pixel & Pine",        "plan": "Free",       "arr_usd": 0,      "renewal_date": None},
    "C-104": {"name": "Meridian Freight",    "plan": "Enterprise", "arr_usd": 310000, "renewal_date": "2026-10-07"},
    "C-105": {"name": "Oakline Studio",      "plan": "Growth",     "arr_usd": 12000,  "renewal_date": "2026-12-01"},
}

In [5]:
from anthropic import Anthropic

client = Anthropic()

SYSTEM_PROMPT = (
    "You triage customer support tickets. The tickets are in "
    "/mnt/session/uploads/tickets.json. For every ticket, call lookup_account "
    "with its customer_id before deciding its priority. Rank all tickets from "
    "most to least urgent, weighing both the problem and the customer's account. "
    "Write the result to /mnt/session/outputs/triage.json as a list of objects "
    "with rank, ticket_id, and a one-sentence reason."
)

LOOKUP_ACCOUNT_TOOL = {
    "type": "custom",
    "name": "lookup_account",
    "description": (
        "Look up a customer's account in the company CRM. Returns the company "
        "name, plan (Free, Growth, or Enterprise), annual revenue in USD, and "
        "renewal date. Use this for every ticket before ranking it, since the "
        "ticket text alone doesn't show how important the customer is. Returns "
        "an error if the customer_id isn't in the CRM."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "customer_id": {"type": "string", "description": "The customer ID from the ticket, like C-101"}
        },
        "required": ["customer_id"],
    },
}

if "agent" not in globals():  # create once; re-running this cell reuses it
    agent = client.beta.agents.create(
        name="Ticket Triage",
        model="claude-opus-5-5",
        system=SYSTEM_PROMPT,
        tools=[{"type": "agent_toolset_20260401"}, LOOKUP_ACCOUNT_TOOL],
    )
print("agent:", agent.id)

agent: agent_01PE2f8KQe1LTiRu6nkzTRQx


In [6]:
if "environment" not in globals():
    environment = client.beta.environments.create(
        name="triage-env",
        config={"type": "cloud", "networking": {"type": "limited"}},  # triage needs no internet
    )
if "tickets_file" not in globals():
    tickets_file = client.files.upload(file=Path("tickets.json"))

print("environment:", environment.id)
print("tickets file:", tickets_file.id)

environment: env_01XbxXKyp2Gsh7qZFV3QuxdG
tickets file: file_01WmT4ak418MmpFmw3NFFdFo


In [7]:
if "sessions" not in globals():
    sessions = []  # every session you start, so step 6 can archive them


def start_session(title, agent_id=None):
    session = client.beta.sessions.create(
        agent=agent_id or agent.id,
        environment_id=environment.id,
        title=title,
        resources=[{"type": "file", "file_id": tickets_file.id, "mount_path": "/tickets.json"}],
    )
    sessions.append(session)
    return session


print("start_session is ready")

start_session is ready


In [8]:
def lookup_account(customer_id):
    if customer_id not in CRM:
        return {"error": f"No account found for customer_id {customer_id}"}
    return CRM[customer_id]


HANDLERS = {"lookup_account": lookup_account}  # tool name -> the function that answers it

In [9]:
def run_turn(session, text):
    """Send one message and handle events until the agent finishes. Returns call and result counts."""
    calls = {}  # custom tool calls seen so far, by event ID
    stats = {"calls": 0, "results_sent": 0}
    answered = set()  # calls we've already sent results for

    def send_results(results):
        client.beta.sessions.events.send(session.id, events=results)
        stats["results_sent"] += len(results)

    with client.beta.sessions.events.stream(session.id) as stream:
        client.beta.sessions.events.send(
            session.id,
            events=[{"type": "user.message", "content": [{"type": "text", "text": text}]}],
        )
        for event in stream:
            helpers.print_event(event)

            if event.type == "agent.custom_tool_use":
                calls[event.id] = event
                stats["calls"] += 1

            elif event.type == "session.status_idle":
                if event.stop_reason and event.stop_reason.type == "requires_action":
                    results = []
                    for event_id in event.stop_reason.event_ids:
                        if event_id in answered:
                            continue  # already answered; the session hadn't processed it yet
                        call = calls[event_id]
                        result = HANDLERS[call.name](**call.input)
                        print(f"  -> answered {call.name}: {result}")
                        results.append({
                            "type": "user.custom_tool_result",
                            "custom_tool_use_id": event_id,
                            "content": [{"type": "text", "text": json.dumps(result)}],
                            "is_error": "error" in result,
                        })
                        answered.add(event_id)
                    if results:
                        send_results(results)
                else:
                    print("\n\nAgent finished.")
                    break
    return stats

## Demo starts here

### 1 · Run the triage

In [10]:
%%time
session = start_session("Demo")
stats = run_turn(session, "Triage the support tickets.")


[bash] {'command': 'cat /mnt/session/uploads/tickets.json'}

[CUSTOM lookup_account] {'customer_id': 'C-101'}

[CUSTOM lookup_account] {'customer_id': 'C-102'}

[CUSTOM lookup_account] {'customer_id': 'C-103'}

[CUSTOM lookup_account] {'customer_id': 'C-104'}

[CUSTOM lookup_account] {'customer_id': 'C-105'}

[CUSTOM lookup_account] {'customer_id': 'C-999'}
  -> answered lookup_account: {'name': 'Northwind Logistics', 'plan': 'Growth', 'arr_usd': 18000, 'renewal_date': '2027-02-01'}
  -> answered lookup_account: {'name': 'Brightpath Health', 'plan': 'Enterprise', 'arr_usd': 240000, 'renewal_date': '2027-03-15'}
  -> answered lookup_account: {'name': 'Pixel & Pine', 'plan': 'Free', 'arr_usd': 0, 'renewal_date': None}
  -> answered lookup_account: {'name': 'Meridian Freight', 'plan': 'Enterprise', 'arr_usd': 310000, 'renewal_date': '2026-10-07'}
  -> answered lookup_account: {'name': 'Oakline Studio', 'plan': 'Growth', 'arr_usd': 12000, 'renewal_date': '2026-12-01'}
  -> answered lookup

### 2 · Read the ranking

In [11]:
triage = json.loads(helpers.download(client, session.id, "triage.json").read_text())
for row in triage:
    print(row["rank"], row["ticket_id"], "-", row["reason"])

1 T-1002 - Complete SSO outage locking out every user at Brightpath Health, an Enterprise account worth $240k ARR.
2 T-1004 - Though labeled minor, the wrong CSV date format will disrupt quarter close next week for Meridian Freight, our largest account ($310k ARR, Enterprise), which renews on 2026-10-07.
3 T-1007 - Intermittent 500 errors on /v2/orders are a serious production issue that could affect other customers too, but customer ID C-999 isn't in the CRM, so the account needs to be verified.
4 T-1006 - Oakline Studio (Growth, $12k ARR) was charged twice, a billing error that needs a refund soon to protect trust ahead of its December renewal.
5 T-1001 - Dashboard load times of 20-30 seconds hurt Northwind Logistics (Growth, $18k ARR), but the product still works.
6 T-1005 - A simple how-to question about inviting a teammate from Northwind Logistics (Growth), quick to answer with a docs link.
7 T-1003 - A cosmetic complaint about dark mode from a Free-plan account with $0 ARR; the a

### 3 · Follow up in the same session

In [12]:
%%time
followup = run_turn(session, "Draft a short reply to each of the top 3 tickets and save them to /mnt/session/outputs/replies.md")
print("\nlookups this turn:", followup["calls"])


[write] {'content': "# Draft Replies: Top 3 Tickets\n\n---\n\n## T-1002: SSO login failing for all users\n**To:** Brightpath Hea
I wrote draft replies to the top 3 tickets and saved them to `/mnt/session/outputs/replies.md`.

- **T-1002 (Brightpath, SSO lockout):** Says we've escalated it to engineering. Asks for the Okta error, when it started and any recent Okta changes, and suggests an admin fallback login to get back in.
- **T-1004 (Meridian, CSV dates):** Treats it as a priority rather than a "minor" issue because of their quarter close. Promises a fix or workaround before the close and asks which exports they use.
- **T-1007 (C-999, API 500s):** Asks for request IDs and timestamps. It also asks for the account email or workspace name, since the customer ID didn't match anything in the CRM.

**Check these promises before sending:** the drafts make a few commitments that aren't confirmed yet. Keep, change or remove each one:
- T-1002 promises an update "within the hour."
- T-1004 

## After the session

In [13]:
for s in sessions:
    try:
        client.beta.sessions.archive(s.id)
        print(s.id, client.beta.sessions.retrieve(s.id).status)
    except Exception as e:
        print(s.id, 'skipped:', e)

# Done presenting for good? Uncomment these. Nothing above will run afterward.
# _ = client.beta.environments.archive(environment.id)
# _ = client.beta.agents.archive(agent.id)

sesn_01WqRXXvU4N91jZtbUdwezJd terminated
